# nanoWxRisk: V0 (VAE pretraining on gridMET 4 km windows) on free Colab

Runtime > Change runtime type > T4 GPU. Upload the folder `colab_upload/` (code.zip, gridmet_win.npy, gridmet_win_days.npy, static.npz; 2.4 GB) to Google Drive at `MyDrive/wxrisk/`. Checkpoints are written to Drive after every epoch; if the session disconnects, run all cells again and training resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, shutil, zipfile
DRIVE = '/content/drive/MyDrive/wxrisk'
LOCAL = '/content/wxrisk_data'
os.makedirs(f'{LOCAL}/cnn', exist_ok=True)
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)

In [ ]:
# local copies: random reads of the memory-mapped windows are too slow over the Drive mount
for f in ['gridmet_win.npy','gridmet_win_days.npy','static.npz']:
    if not os.path.exists(f'{LOCAL}/cnn/{f}'):
        shutil.copy(f'{DRIVE}/{f}', f'{LOCAL}/cnn/{f}')
if not os.path.exists('/content/code'):
    zipfile.ZipFile(f'{DRIVE}/code.zip').extractall('/content/code')
print(os.listdir(f'{LOCAL}/cnn'), os.listdir('/content/code'))

In [ ]:
# smoke test (1 minute): the pipeline, device and checkpoint
!cd /content/code && WXRISK_DATA={LOCAL} python scripts/82_latent_v0.py train --tag smoke --epochs 1 --smoke 64 --ckpt-dir /content/smoke_ck 2>&1 | grep -v Warn | tail -5

In [ ]:
# V0 pretraining, windows 2000-2016, 8 latent channels per 0.25 deg cell, 12 epochs; resumes from the Drive checkpoint
!cd /content/code && WXRISK_DATA={LOCAL} python scripts/82_latent_v0.py train --tag c8_2000 --c 8 --from-year 2000 --epochs 12 --ckpt-dir {DRIVE}/ckpt 2>&1 | grep -v Warn

In [ ]:
# the data-scale comparison: pretraining windows 2010-2016 only
!cd /content/code && WXRISK_DATA={LOCAL} python scripts/82_latent_v0.py train --tag c8_2010 --c 8 --from-year 2010 --epochs 12 --ckpt-dir {DRIVE}/ckpt 2>&1 | grep -v Warn

In [ ]:
# keep the logs next to the models
import glob
for f in glob.glob(f'{LOCAL}/audit/*.txt'):
    shutil.copy(f, f'{DRIVE}/ckpt/')
print(os.listdir(f'{DRIVE}/ckpt'))

Back on the Mac: copy `vae_c8_2000.pt` and `vae_c8_2010.pt` from Drive into `~/wxrisk_data/cnn/`, then `python scripts/82_latent_v0.py probe --tag c8_2000`.